Description :

This notebook converts cleaned text chunks into vector embeddings and stores them in a FAISS vector index. The vector store enables efficient semantic similarity search, which is the core retrieval mechanism of the RAG system.

Key tasks performed:

Load cleaned text chunks

Generate embeddings using Sentence Transformers

Build and persist a FAISS vector index

Store metadata for traceability and retrieval

In [4]:
%pip install sentence-transformers faiss-cpu

  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached sentence_transformers-5.2.0-py3-none-any.whl (493 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   --- ------------------------------------ 1.6/18.9 MB 7.6 MB/s eta 0:00:03
   ------- -------------------------------- 3.4/18.9 MB 8.1 MB/s eta 0:00:02
   ------------ --------------------------- 5.8/18.9 MB 9.0 MB/s eta 0:00:02
   ------------------- -------------------- 9.2/18.9 MB 10.8 MB/s eta 0:00:01
   ----------------------- ---------------- 11.0/18.9 MB 10.7 MB/s eta 0:00:01
   --------------------------- ------------ 13.1/18.9 MB 10.

In [5]:
import json

with open("Data/railway_rules_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))

Total chunks: 23


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\KIIT0001\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KIIT0001\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
texts = [c["text"] for c in chunks]

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(23, 384)


In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 23


In [9]:
faiss.write_index(index, "Data/railway_faiss.index")

with open("Data/railway_metadata.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Vector store saved")

Vector store saved


In [10]:
query = "What are passenger rights in Indian Railways?"
query_embedding = model.encode([query])

D, I = index.search(query_embedding, k=3)

for idx in I[0]:
    print("----")
    print(chunks[idx]["text"][:300])

----
(xiii) People should not trespass on Railway tracks. (xiv) Security Help Line no.139 may also be used to inform regarding suspected objects/ persons on trains/Railway Premises. Note: 1. The above time limits are applicable to Railways for delivering various services, provided the customer has fulfil
----
322 Citizens’ Charter on Passenger Services of Indian Railways Preamble This Charter is a commitment of the Indian Railway Administration to: • Provide safe and dependable train services • Set notified standards for various services, wherever possible • Ensure adequate passenger amenities in trains 
----
help line No. 139, while on board or at station. 8 Theft of luggage (i) A Prescribed FIR form is available in the time table or with TTEs/Guards and RPF/GRP escorts. After filing it up, the form may be handed over to one of the officials viz, TTE, Guard or RPF/GRP escort for registration of the repo
